# An Introduction to Git

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

An *interactive* guide: every command below runs in a small sandbox repository
created by the notebook itself.

## What you will learn

- What version control is and why you need it
- The three places a file can be: working directory, staging area, repository
- The everyday cycle: `status` &rarr; `add` &rarr; `commit`
- Looking at history and at differences
- Undoing things
- Branches, merges, and merge conflicts
- Working with GitHub: `clone`, `push`, `pull`, forks and pull requests

## Why version control?

Without it, a project ends up looking like this:

```
thesis_v2.tex   thesis_v2_final.tex   thesis_v2_final_REAL.tex
analysis.py     analysis_old.py       analysis_works.py
```

Git gives you instead:

- A **complete history**: every version, who changed what, and why.
- The ability to **go back** to any earlier state.
- A way for several people to work on the same files **without overwriting each other**.
- A **backup**, once the project is on a server such as GitHub.

💡 Think of Git as a *time machine* for your project.

## The model: three places

A file in a Git project lives in one of three places.

```
   working directory  --- git add --->  staging area  --- git commit --->  repository
   (what you edit)                      (what will go                     (the permanent
                                         in the next commit)               history)
```

- The **working directory** is the folder you actually edit.
- The **staging area** (or *index*) is where you assemble the next commit.
- The **repository** is the `.git` folder: the full history, stored locally.

A **commit** is a snapshot of the whole project, with a message, an author and a date.

## A sandbox to play in

Everything happens inside `git_sandbox`, so nothing else on your computer is touched.
Running this cell again starts from a clean state.

In [ ]:
import os, shutil

BASE = globals().get("BASE", os.getcwd())
os.chdir(BASE)
shutil.rmtree("git_sandbox", ignore_errors=True)
os.makedirs("git_sandbox")
os.chdir("git_sandbox")
print("Working inside:", os.path.basename(os.getcwd()) + "/")

## Who are you?

Every commit records an author. You set this **once per computer**:

```bash
git config --global user.name  "Your Name"
git config --global user.email "your@email.com"
```

Use the same email as your GitHub account, so that your commits are linked to it.

Without `--global`, the setting applies **only to the current repository** &mdash;
which is what we do below, so that this notebook does not touch your real configuration.

## Creating a repository

`git init` turns an ordinary folder into a repository. It creates a hidden `.git`
folder; everything else stays exactly as it was.

The `-b main` option names the first branch `main`. Without it, older versions of
Git call it `master`, which is a common source of confusion.

In [ ]:
!git init -b main
!git config user.name  "FCA Student"
!git config user.email "student@usc.es"
!ls -a

## The everyday cycle

1. `git status` &mdash; what has changed?
2. `git add file` &mdash; put it in the staging area
3. `git commit -m "message"` &mdash; record it in the history

Do this often. Small, frequent commits are far more useful than large, rare ones.

In [ ]:
!echo 'print("Hello Git")' > hello.py
!git status

Git calls `hello.py` **untracked**: it has never been committed.

In [ ]:
!git add hello.py
!git status --short

`A` means *added to the staging area*. Now we record it.

In [ ]:
!git commit -m "Add hello.py, a first script" 

### Writing good commit messages

- One short line (about 50 characters), in the imperative: *"Add"*, *"Fix"*, *"Remove"*.
- Say **why**, not what: the diff already says what changed.
- One commit = one idea. Do not mix a bug fix with a reformatting.

Bad: `changes`, `stuff`, `asdf`, `final version 3`.

## Looking at the history

```bash
git log                      # full history
git log --oneline            # one line per commit
git log --oneline --graph    # with the branch structure
git show HEAD                # what the last commit changed
```

`HEAD` is a pointer to where you are now, normally the last commit of the current branch.

In [ ]:
!git log --oneline

## Seeing what changed: `git diff`

| Command | Compares |
|---|---|
| `git diff` | working directory &harr; staging area (*not yet added*) |
| `git diff --staged` | staging area &harr; last commit (*added, not yet committed*) |
| `git diff HEAD~1 HEAD` | between two commits |

Read a diff as: `-` lines removed, `+` lines added.

In [ ]:
!echo 'print("Goodbye Git")' >> hello.py
!git diff

In [ ]:
!git add hello.py
!git commit -m "Add a farewell message" 

## Ignoring files: `.gitignore`

Never commit compiled output, data files, or notebook checkpoints.
List the patterns to ignore in a file called `.gitignore`:

```
__pycache__/
*.pyc
.ipynb_checkpoints/
*.root
data/
.DS_Store
```

`.gitignore` itself **is** committed, so everyone on the project ignores the same things.

In [ ]:
%%writefile .gitignore
__pycache__/
*.pyc
.ipynb_checkpoints/
data/

In [ ]:
!mkdir -p data && echo "10 20 30" > data/measurements.txt
!git status --short
!git add .gitignore && git commit -q -m "Add .gitignore"
!echo "(data/ is ignored, so it does not appear above)" 

## Undoing things

| Situation | Command |
|---|---|
| Discard changes in a file (not yet added) | `git restore file` |
| Take a file out of the staging area | `git restore --staged file` |
| Change the last commit message | `git commit --amend` |
| Undo a commit, keeping the changes | `git reset --soft HEAD~1` |
| Undo a commit **and** the changes | `git reset --hard HEAD~1` ⚠️ |
| Undo a commit by making a new one | `git revert HEAD` |

> ⚠️ `reset --hard` throws work away permanently. `revert` is the safe option,
> and the only correct one if the commit has already been pushed.

In [ ]:
!echo "a mistake" >> hello.py
!git status --short
!echo "(restoring)" 
!git restore hello.py
!git status --short
!echo "(clean again)" 

## Branches

A branch is an independent line of work. You develop a new idea on its own branch,
and only merge it into `main` when it works.

```bash
git branch                 # list branches
git switch -c feature      # create one and move to it  (modern)
git checkout -b feature    # the same, older syntax
git switch main            # go back
```

💡 Branches are *parallel universes* of your project. `main` should always work.

In [ ]:
!git switch -c feature
!echo 'def energy(m): return m * 299792458**2' > physics.py
!git add physics.py
!git commit -q -m "Add energy function"
!git log --oneline --graph --all

## Merging

`git merge other` brings the work of `other` into the branch you are on.
So: **first move to the destination**, then merge.

`--no-edit` accepts the default merge message. Without it Git opens an editor,
and inside a notebook that would hang the kernel.

There are **two possible outcomes**. Telling them apart is the whole point.


In [ ]:
!git switch main
!git merge feature --no-edit
!ls
!git log --oneline --graph --all

### Case 1: fast-forward

Git printed **`Fast-forward`**, and the history is a straight line:
there is **no merge commit**.

Why? `main` had not moved since `feature` branched off, so `feature` already
contained every commit of `main`. Nothing had to be combined &mdash; Git simply slid
the `main` pointer forward.


### Case 2: the branches diverge

Now let **both** sides move. Branch `plots` adds a file, and `main` adds a
different one *after* `plots` was created. The two histories genuinely fork,
so a fast-forward is no longer possible.


In [ ]:
!git switch -c plots -q
!echo 'import matplotlib.pyplot as plt' > plotting.py
!git add plotting.py && git commit -q -m "Add plotting helper"

!git switch main -q
!echo '# project notes' > NOTES.md
!git add NOTES.md && git commit -q -m "Add project notes"

!git merge plots --no-edit
!git log --oneline --graph --all


This time Git writes a **merge commit**: a commit with *two* parents.
The graph shows the fork and the join.

```
*   Merge branch 'plots'
|\
| * Add plotting helper
* | Add project notes
|/
```

Rule of thumb: **fast-forward** when the branches never diverged, **merge commit**
when they did. Adding `--no-ff` forces a merge commit even in case 1, so the history
keeps a visible record that a branch ever existed.


## Merge conflicts

A conflict happens when **the same lines** were changed on both branches.
Git cannot decide for you, so it stops and asks.

This is normal. It is not an error, and it is not a disaster.

In [ ]:
# two branches change the same line
!git switch -c experiment -q
!echo 'C = 3.0e8   # rounded' > constants.py
!git add constants.py && git commit -q -m "Add rounded constant"

!git switch main -q
!echo 'C = 299792458   # exact, m/s' > constants.py
!git add constants.py && git commit -q -m "Add exact constant"

!git merge experiment

Git leaves **markers** in the file showing both versions:

```
<<<<<<< HEAD
your version
=======
their version
>>>>>>> experiment
```

To resolve: edit the file so it says what you want, **delete the markers**,
then `git add` it and commit.

In [ ]:
!cat constants.py

In [ ]:
# resolve by hand: keep the exact value, and say so
!echo 'C = 299792458   # exact speed of light, m/s' > constants.py
!git add constants.py
!git commit -m "Merge experiment: keep the exact value of c"
!git log --oneline --graph --all

## Working with a remote: GitHub

A **remote** is a copy of the repository on a server.

```bash
git clone URL                    # get a copy of an existing repository
git remote -v                    # which remotes do I have?
git remote add origin URL        # connect a local repo to a remote
git push -u origin main          # send your commits (first time)
git push                         # afterwards
git pull                         # bring in what others have pushed
```

`origin` is just the conventional name of the main remote.

### `git clone` works with any path, not only a URL

Below we clone our sandbox repository into a second folder, to show what a clone is:
a **complete, independent copy**, history included.

In [ ]:
import os
os.chdir(BASE)
!git clone -q git_sandbox git_clone
os.chdir("git_clone")
!git log --oneline | head -3
!git remote -v

In [ ]:
import os
os.chdir(os.path.join(BASE, "git_sandbox"))
print("back in:", os.path.basename(os.getcwd()) + "/")

## Authenticating with GitHub

⚠️ **GitHub stopped accepting your account password over HTTPS in 2021.**
The first time you push you will need one of these:

| Method | How |
|---|---|
| **Personal access token** | GitHub &rarr; Settings &rarr; Developer settings &rarr; Tokens. Use the token *instead of the password* when Git asks. |
| **SSH key** | `ssh-keygen -t ed25519 -C "your@email.com"`, then paste `~/.ssh/id_ed25519.pub` into GitHub &rarr; Settings &rarr; SSH keys. Clone with the `git@github.com:...` URL. |

The SSH key is more work once, and no work ever again. Set it up today, not the day you need it.

## Collaborating: fork, branch, pull request

The workflow used by essentially every project you will contribute to:

1. **Fork** the repository on GitHub &mdash; your own copy of it.
2. **Clone** your fork to your computer.
3. Create a **branch** for what you want to do.
4. Commit, and **push** the branch to your fork.
5. Open a **pull request**: *"please take these commits into your project"*.
6. Others **review** it, you fix what they raise, and it gets **merged**.

The point of a pull request is the review. It is where most of the learning happens.

## Reading the history of a project

| Where | How |
|---|---|
| Terminal | `git log --oneline --graph --all` |
| GitHub | the *Commits* tab, or *Insights &rarr; Network* |
| Editor | VS Code with the *Git Graph* extension |
| Dedicated tools | GitKraken, Sourcetree |

Useful when you arrive at a project you did not write: `git log -p file.py` shows
the whole history of one file, and `git blame file.py` shows who last touched each line.

In [ ]:
!git log --oneline --graph --all

## Cleaning up

Everything we made lives in `git_sandbox` and `git_clone`. This removes both.

> Run this **only when you are done**. Afterwards the sandboxes are gone, and the
> cells above will fail until you re-run the setup cell that created them.

In [ ]:
import os, shutil

BASE = globals().get("BASE", os.getcwd())      # also works after a kernel restart
os.chdir(BASE)
for d in ("git_sandbox", "git_clone"):
    shutil.rmtree(d, ignore_errors=True)
print("Sandboxes removed. Back in:", os.path.basename(os.getcwd()) + "/")

## Exercise

Work in pairs.

1. Create a repository on GitHub and clone it.
2. Add a module with a function (for example, a numerical integrator) and a second
   module with a test for it. Commit and push.
3. **Fork** the repository of the pair next to you, and clone your fork.
4. On a branch, add one more function *and its test*.
5. Push the branch and open a **pull request**.
6. Review the pull request you receive: check that the tests pass and that the
   commit messages say something. Ask for a change if they do not.
7. Merge it.
8. Back on your own clone, `git pull` and check that you have their work.

Bonus: make two people change the same line on purpose, and resolve the conflict.

## Cheat sheet

| | |
|---|---|
| `git init -b main` | start a repository |
| `git clone URL` | copy an existing one |
| `git status` | what has changed? |
| `git add file` &middot; `git add .` | stage changes |
| `git commit -m "..."` | record them |
| `git log --oneline --graph --all` | see the history |
| `git diff` &middot; `git diff --staged` | see the changes |
| `git restore file` | discard changes |
| `git switch -c name` &middot; `git switch main` | create / change branch |
| `git merge other` | bring another branch in |
| `git push` &middot; `git pull` | exchange with the remote |
| `.gitignore` | what never gets committed |

**The cycle you will use every day:** `status` &rarr; `add` &rarr; `commit` &rarr; `push`.